# INUA Sprint 1: Synthetic Dataset Generator

**Student:** Michelle  
**Project:** INUA - Intelligent Mentorship Platform  
**Institution:** Strathmore University (Final Year Project - BICS Year 4)  
**Document Reference:** `ml/data/data_dictionary.md`  
**Deliverable:** `ml/data/inua_mentorship_dataset.csv` (2,000 sessions, 22 features)  

---

## Academic Foundation
This notebook implements the research-backed generation engine for the INUA project. Rather than using naive stochastic noise, distributions and interdependencies are grounded in:
- **Kathy Kram Mentoring Theory (1985)**: Disentangling instrumental Career Support from emotional Psychosocial Support.
- **Albert Bandura Self-Efficacy Theory (1997)**: Modeling Confidence Growth as a function of psychological safety and goal attainment.
- **Allen et al. (2004) Meta-Analysis**: Empirical non-linear correlation between mentor professional seniority and mentorship quality.

### Step 1: Environment Setup & Seed Initialization
Fixing a random seed (`seed = 42`) guarantees 100% reproducibility across different execution environments.

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd

# Set global deterministic seed
SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)
print(f"[INUA] Random seed initialized to: {SEED}")

### Step 2: Mentee Profile Generation
Generates `career_stage`, `field_of_study`, `career_interest`, and `prior_sessions` using the probability weights defined in the Master Data Dictionary.

In [ ]:
def generate_student_profile(n_samples: int, rng: np.random.Generator) -> pd.DataFrame:
    career_stages = ["Year1", "Year2", "Year3", "Year4", "Graduate", "EarlyCareer"]
    stage_probs = [0.125, 0.160, 0.210, 0.240, 0.140, 0.125]

    fields_of_study = [
        "Computer Science", "Engineering", "Business", "Data Analytics",
        "Health Sciences", "Education", "Social Sciences", "Other STEM"
    ]
    field_probs = [0.21, 0.16, 0.15, 0.11, 0.09, 0.07, 0.06, 0.15]

    career_interests = [
        "Data Science", "Software Engineering", "Cybersecurity", "Artificial Intelligence",
        "Cloud Computing", "UI/UX", "Product Management", "Finance", "Entrepreneurship"
    ]
    interest_probs = [0.17, 0.18, 0.11, 0.13, 0.10, 0.09, 0.08, 0.08, 0.06]

    stage_col = rng.choice(career_stages, size=n_samples, p=stage_probs)
    field_col = rng.choice(fields_of_study, size=n_samples, p=field_probs)
    interest_col = rng.choice(career_interests, size=n_samples, p=interest_probs)
    prior_col = np.clip(rng.poisson(lam=2.2, size=n_samples), 0, 12)

    return pd.DataFrame({
        "career_stage": stage_col,
        "field_of_study": field_col,
        "career_interest": interest_col,
        "prior_sessions": prior_col,
    })

### Step 3: Mentor Profile Generation
Generates `mentor_experience_years` (Gamma distribution), `mentor_industry`, `mentor_specialization`, and `mentor_availability`.

In [ ]:
def generate_mentor_profile(n_samples: int, rng: np.random.Generator) -> pd.DataFrame:
    industries = [
        "Technology", "Banking & Finance", "Academia",
        "Healthcare", "NGO", "Government", "Entrepreneurship"
    ]
    industry_probs = [0.28, 0.16, 0.14, 0.10, 0.10, 0.08, 0.14]
    availabilities = ["High", "Medium", "Low"]
    availability_probs = [0.35, 0.45, 0.20]

    specialization_map = {
        "Technology": ["Cloud Architecture", "Backend Engineering", "AI Systems", "Cyber Defense", "Full Stack Dev"],
        "Banking & Finance": ["Financial Modeling", "Fintech Innovations", "Risk Management", "Investment Banking"],
        "Academia": ["Postgraduate Research", "Scientific Writing", "Doctoral Pathways", "Applied Statistics"],
        "Healthcare": ["Health Informatics", "Biomedical Systems", "Public Health Research"],
        "NGO": ["Program Evaluation", "Development Policy", "Sustainable Tech"],
        "Government": ["Public Sector ICT", "E-Governance", "Digital Infrastructure"],
        "Entrepreneurship": ["Tech Venture Building", "Product Strategy", "Seed Fundraising", "Go-to-Market"],
    }

    chosen_ind = rng.choice(industries, size=n_samples, p=industry_probs)
    chosen_spec = [rng.choice(specialization_map[ind]) for ind in chosen_ind]
    exp_col = np.clip(np.round(rng.gamma(shape=2.8, scale=3.2, size=n_samples)).astype(int), 1, 25)
    avail_col = rng.choice(availabilities, size=n_samples, p=availability_probs)

    return pd.DataFrame({
        "mentor_experience_years": exp_col,
        "mentor_industry": chosen_ind,
        "mentor_specialization": chosen_spec,
        "mentor_availability": avail_col,
    })

### Step 4: Session Mechanics Generation (Rules 1 & 2)
- **Rule 1**: Student career stage conditions session type and developmental goal category.
- **Rule 2**: Session type dictates realistic session duration (minutes).

In [ ]:
def generate_session(students_df: pd.DataFrame, mentors_df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    n_samples = len(students_df)
    session_ids = [f"INUA_SESS_{i+1:04d}" for i in range(n_samples)]

    session_types = ["Career Guidance", "CV Review", "Interview Preparation", "Technical Coaching", "Networking"]
    stage_to_session_probs = {
        "Year1": [0.50, 0.15, 0.05, 0.05, 0.25],
        "Year2": [0.35, 0.35, 0.05, 0.15, 0.10],
        "Year3": [0.10, 0.30, 0.20, 0.35, 0.05],
        "Year4": [0.10, 0.25, 0.40, 0.20, 0.05],
        "Graduate": [0.15, 0.20, 0.45, 0.10, 0.10],
        "EarlyCareer": [0.25, 0.10, 0.20, 0.20, 0.25],
    }

    goal_categories = ["Internship", "Portfolio Development", "Technical Skills", "Career Transition", "Leadership", "Research"]
    stage_to_goal_probs = {
        "Year1": [0.15, 0.15, 0.30, 0.10, 0.15, 0.15],
        "Year2": [0.35, 0.25, 0.25, 0.05, 0.05, 0.05],
        "Year3": [0.45, 0.25, 0.20, 0.03, 0.03, 0.04],
        "Year4": [0.30, 0.30, 0.20, 0.10, 0.05, 0.05],
        "Graduate": [0.10, 0.25, 0.15, 0.35, 0.10, 0.05],
        "EarlyCareer": [0.05, 0.15, 0.15, 0.35, 0.25, 0.05],
    }

    duration_params = {
        "Career Guidance": {"mean": 55, "std": 8, "min": 40, "max": 70},
        "CV Review": {"mean": 45, "std": 7, "min": 30, "max": 60},
        "Interview Preparation": {"mean": 65, "std": 10, "min": 45, "max": 90},
        "Technical Coaching": {"mean": 85, "std": 15, "min": 60, "max": 120},
        "Networking": {"mean": 30, "std": 6, "min": 20, "max": 45},
    }

    chosen_types, chosen_goals, durations = [], [], []
    for stage in students_df["career_stage"]:
        st = rng.choice(session_types, p=stage_to_session_probs[stage])
        g = rng.choice(goal_categories, p=stage_to_goal_probs[stage])
        cfg = duration_params[st]
        d = int(np.clip(rng.normal(cfg["mean"], cfg["std"]), cfg["min"], cfg["max"]))
        chosen_types.append(st)
        chosen_goals.append(g)
        durations.append(d)

    attendance_opts = ["Attended", "Rescheduled", "Cancelled"]
    att_probs = [0.80, 0.13, 0.07]
    platforms = ["Teams", "Zoom", "Meet", "Physical"]
    plat_probs = [0.35, 0.30, 0.20, 0.15]

    return pd.DataFrame({
        "session_id": session_ids,
        "session_type": chosen_types,
        "session_duration_minutes": durations,
        "attendance_status": rng.choice(attendance_opts, size=n_samples, p=att_probs),
        "meeting_platform": rng.choice(platforms, size=n_samples, p=plat_probs),
        "goal_category": chosen_goals,
    })

### Step 5: Correlated Likert Quality Scores (Rules 3, 4, 5)
- **Rule 3**: Mentor Experience -> Career Support via logarithmic progression
- **Rule 4**: Mentor Availability -> Mentor Responsiveness
- **Rule 5**: Communication Quality -> Psychosocial Support

In [ ]:
def generate_scores(sessions_df: pd.DataFrame, mentors_df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    n_samples = len(sessions_df)
    career_support, psychosocial, communication, goal_prog, confidence, responsiveness = [], [], [], [], [], []

    avail_dist = {
        "High": ([3, 4, 5], [0.10, 0.45, 0.45]),
        "Medium": ([2, 3, 4, 5], [0.15, 0.50, 0.30, 0.05]),
        "Low": ([1, 2, 3, 4], [0.35, 0.40, 0.20, 0.05]),
    }

    for i in range(n_samples):
        att = sessions_df.iloc[i]["attendance_status"]
        exp = mentors_df.iloc[i]["mentor_experience_years"]
        avail = mentors_df.iloc[i]["mentor_availability"]

        if att == "Cancelled":
            career_support.append(1)
            psychosocial.append(1)
            communication.append(1)
            goal_prog.append(1)
            confidence.append(1)
            responsiveness.append(1)
            continue

        # Rule 4: Responsiveness
        vals, probs = avail_dist[avail]
        resp = int(rng.choice(vals, p=probs))

        # Rule 3: Career Support (Logarithmic model)
        mu_car = 1.8 + 0.95 * math.log(exp + 1.0)
        car = int(np.clip(round(rng.normal(mu_car, 0.45)), 1, 5))

        # Communication Quality
        if resp >= 4:
            comm = int(rng.choice([3, 4, 5], p=[0.20, 0.45, 0.35]))
        elif resp == 3:
            comm = int(rng.choice([2, 3, 4, 5], p=[0.15, 0.50, 0.30, 0.05]))
        else:
            comm = int(rng.choice([1, 2, 3, 4], p=[0.35, 0.40, 0.20, 0.05]))

        # Rule 5: Psychosocial Support
        mu_psy = 0.3 + 0.90 * comm
        psy = int(np.clip(round(rng.normal(mu_psy, 0.40)), 1, 5))

        # Goal Progress & Confidence Growth
        gp = int(np.clip(round(rng.normal(0.50 * car + 0.30 * comm + 0.20 * resp, 0.45)), 1, 5))
        conf = int(np.clip(round(rng.normal(0.55 * psy + 0.35 * gp + 0.10 * comm, 0.40)), 1, 5))

        if att == "Rescheduled":
            resp = max(1, resp - int(rng.choice([0, 1], p=[0.4, 0.6])))
            gp = max(1, gp - int(rng.choice([0, 1], p=[0.5, 0.5])))
            conf = max(1, conf - int(rng.choice([0, 1], p=[0.6, 0.4])))

        career_support.append(car)
        psychosocial.append(psy)
        communication.append(comm)
        goal_prog.append(gp)
        confidence.append(conf)
        responsiveness.append(resp)

    return pd.DataFrame({
        "career_support_score": career_support,
        "psychosocial_support_score": psychosocial,
        "communication_quality": communication,
        "goal_progress_score": goal_prog,
        "confidence_growth_score": confidence,
        "mentor_responsiveness_score": responsiveness,
    })

### Step 6 & 7: Effectiveness Index & Target Label (Rule 6)
Computes multi-attribute continuous `effectiveness_index` and ground-truth binary label `session_effectiveness`.

In [ ]:
def calculate_effectiveness(sessions_df: pd.DataFrame, scores_df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    n_samples = len(sessions_df)
    indices, labels = [], []
    w_goal, w_car, w_conf, w_comm, w_psy, w_resp = 0.25, 0.20, 0.20, 0.15, 0.10, 0.10

    for i in range(n_samples):
        att = sessions_df.iloc[i]["attendance_status"]
        if att == "Cancelled":
            indices.append(round(float(rng.uniform(0.05, 0.22)), 4))
            labels.append(0)
        else:
            sc = scores_df.iloc[i]
            composite = (
                w_goal * (sc["goal_progress_score"] / 5.0) +
                w_car * (sc["career_support_score"] / 5.0) +
                w_conf * (sc["confidence_growth_score"] / 5.0) +
                w_comm * (sc["communication_quality"] / 5.0) +
                w_psy * (sc["psychosocial_support_score"] / 5.0) +
                w_resp * (sc["mentor_responsiveness_score"] / 5.0)
            )
            idx = round(float(np.clip(composite + rng.normal(0, 0.02), 0.10, 0.99)), 4)
            threshold = 0.65 if att == "Attended" else 0.72
            lbl = 1 if idx >= threshold else 0
            indices.append(idx)
            labels.append(lbl)

    return pd.DataFrame({
        "effectiveness_index": indices,
        "session_effectiveness": labels,
    })

### Step 8: Execution, Verification & CSV Export
Assembles the complete 2,000-row DataFrame, validates column order and missing values, and saves `ml/data/inua_mentorship_dataset.csv`.

In [ ]:
N = 2000
students = generate_student_profile(N, rng)
mentors = generate_mentor_profile(N, rng)
sessions = generate_session(students, mentors, rng)
scores = generate_scores(sessions, mentors, rng)
outcomes = calculate_effectiveness(sessions, scores, rng)

column_order = [
    "session_id", "career_stage", "field_of_study", "career_interest", "prior_sessions",
    "mentor_experience_years", "mentor_industry", "mentor_specialization", "mentor_availability",
    "session_type", "session_duration_minutes", "attendance_status", "meeting_platform",
    "goal_category", "career_support_score", "psychosocial_support_score",
    "communication_quality", "goal_progress_score", "confidence_growth_score",
    "mentor_responsiveness_score", "effectiveness_index", "session_effectiveness"
]

df = pd.concat([sessions, students, mentors, scores, outcomes], axis=1)[column_order]

# Verification Assertions
assert len(df) == N, f"Expected {N} rows, got {len(df)}"
assert len(df.columns) == 22, f"Expected 22 columns, got {len(df.columns)}"
assert df.isnull().sum().sum() == 0, "Found unexpected missing values!"

output_csv = os.path.join("..", "data", "inua_mentorship_dataset.csv")
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df.to_csv(output_csv, index=False)

print(f"[INUA] Successfully exported {len(df)} rows to {output_csv}!")
print("\n--- Target Distribution ---")
print(df["session_effectiveness"].value_counts(normalize=True).round(3))
df.head()